# Demo 6 — Stessi clip, metriche e audio degli errori

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

**Risultati registrati:** questa demo esplora il manifest e le predizioni dell’esperimento già eseguito. Non addestra la CNN. Le metriche e i grafici vengono ricalcolati nel browser; i clip degli errori sono scaricati solo su richiesta.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
modello = "CNN"                  # "Dummy", "RF media", "RF media+std", "CNN"
normalizza = True                # True: frazioni per riga; False: conteggi
classe_reale = None              # None: tutte; oppure un id da 0 a 9
indice_errore = 0                # indice nell'elenco degli errori filtrati
risultati = al.load_classification_results(DEMO_ASSETS / "classification-results.json")
assert modello in risultati["models"]
righe = risultati["splits"]["test"]
veri = np.array([r["label"] for r in righe])
for nome,m in risultati["models"].items():
    misure = al.classification_metrics(veri,m["test_predictions"])
    print(f"{nome:14s} accuratezza {misure['accuracy']:.3f}; macro-F1 {misure['macro_f1']:.3f}")
predetti = np.array(risultati["models"][modello]["test_predictions"])
cm = al.classification_metrics(veri,predetti)["confusion"]
valori = np.divide(cm,cm.sum(1,keepdims=True),out=np.zeros_like(cm,dtype=float),where=cm.sum(1,keepdims=True)>0) if normalizza else cm
fig = go.Figure(go.Heatmap(x=risultati["classes"],y=risultati["classes"],z=valori,
    text=cm,texttemplate="%{text}",colorscale="Blues",zmin=0,zmax=1 if normalizza else None,
    hovertemplate="Vera: %{y}<br>Predetta: %{x}<br>Valore: %{z:.3f}<br>Clip: %{text}<extra></extra>"))
fig.update_layout(title=f"{modello} — test registrato, "+("righe normalizzate" if normalizza else "conteggi"),
    xaxis_title="Classe predetta",yaxis_title="Classe reale",yaxis_autorange="reversed",height=580)
fig.show()
maschera = predetti!=veri
if classe_reale is not None:
    assert 0<=classe_reale<10
    maschera &= veri==classe_reale
errori = np.flatnonzero(maschera)
print("Errori disponibili con questo filtro:",len(errori))
for j,k in enumerate(errori[:15]):
    print(j,Path(righe[k]["file"]).name,"vero:",risultati["classes"][veri[k]],"predetto:",risultati["classes"][predetti[k]])
if len(errori):
    assert 0<=indice_errore<len(errori), "Scegliere un indice nell'elenco degli errori."
    k = errori[indice_errore]
    print("Ascolto errore",indice_errore,"di",len(errori),"—",righe[k]["file"])
    print("Vero:",risultati["classes"][veri[k]],"— predetto:",risultati["classes"][predetti[k]])
    y,sr = await al.load_result_clip(righe[k],DEMO_ASSETS/"ch4-audio",base_url=ASSET_BASE+"/ch4-audio" if ASSET_BASE else None)
    display(al.safe_audio(y,sr,normalize=False))
else:
    print("Nessun errore con il filtro selezionato.")
print("Il cambio di modello o filtro esplora risultati registrati: non produce nuove predizioni.")
